<pre>
- Dióxido de nitrogênio   -> Unidade de medida de retorno kg/kg-¹
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.



In [1]:
import cdsapi
import os, sys
import xarray as xr
import zipfile
from pyspark.sql import functions as F

In [2]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
dataset = "cams-global-reanalysis-eac4"
request = {
    "variable": [
        "nitrogen_dioxide"
    ],
    "pressure_level": ["1000"],
    "date": ["2025-12-01/2025-12-31"],
    "time": ["06:00"],
    "data_format": "netcdf",
    "area": [6, -74, -35, -34]
}

client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

ret_download = client.retrieve(dataset, request).download()

print(f"Download completed: {ret_download}")

2026-07-22 15:47:02,121 INFO Request ID is 23c47a6d-4ff0-47cb-baf7-9903d0f9bff0
2026-07-22 15:47:02,305 INFO status has been updated to accepted
2026-07-22 15:47:17,086 INFO status has been updated to running
2026-07-22 15:47:25,155 INFO status has been updated to successful
                                                                                      

Download completed: 17c1ca53856126c7563d4ce3c53f53df.nc


In [4]:
with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Poluicao\\{ret_download}"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:
    # Transforma o Dataset em um Spark Dataframe
    df_dask               = ds.to_dask_dataframe()
    df_dask_c             = df_dask.compute()
    df_dioxido_nitrogenio = spark.createDataFrame(df_dask_c)

df_dioxido_nitrogenio.printSchema()
df_dioxido_nitrogenio.show(10, False)

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- valid_time: timestamp (nullable = true)
 |-- pressure_level: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- no2: float (nullable = true)

+-------------------+--------------+--------+---------+------------+
|valid_time         |pressure_level|latitude|longitude|no2         |
+-------------------+--------------+--------+---------+------------+
|2025-12-01 06:00:00|1000.0        |6.0     |-73.5    |1.2200114E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-72.75   |1.0417561E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-72.0    |3.096072E-9 |
|2025-12-01 06:00:00|1000.0        |6.0     |-71.25   |8.847778E-10|
|2025-12-01 06:00:00|1000.0        |6.0     |-70.5    |4.72933E-10 |
|2025-12-01 06:00:00|1000.0        |6.0     |-69.75   |1.2349333E-9|
|2025-12-01 06:00:00|1000.0        |6.0     |-69.0    |2.0095243E-9|
|2025-12-01 06:00:00|1000.0        |6.0     |-68.25   |1.8070623E-9|
|2025-12-01 06:00:00|1000.0  

In [5]:
# Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)

M_AR = 28.9644 # g/mol
M_NO2 = 46.0055 # g/mol
FATOR_CONVERSAO = (M_AR / M_NO2) * 1e9  # ~ 1.03407e9

drop_cols = ["valid_time", "pressure_level", "no2"]

df_dioxido_nitrogenio_ppb = \
    (df_dioxido_nitrogenio
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("Poluição do ar - NO₂ (ppb)") 
                     ,"valor": (F.col("no2") * F.lit(FATOR_CONVERSAO)).cast("double")
                     ,"unidade_medida": F.lit("ppb")})
         .drop(*drop_cols)

    )

In [6]:
# df_dioxido_nitrogenio_ppb.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_nitrogenio.csv", index=False)

df_dioxido_nitrogenio_ppb.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_nitrogenio.parquet")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [7]:
df_dioxido_nitrogenio_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_nitrogenio.parquet")

df_dioxido_nitrogenio_parquet.printSchema()
df_dioxido_nitrogenio_parquet.show(10, False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+--------+---------+------------+--------------------------+------------------+--------------+
|latitude|longitude|data_medicao|indicador                 |valor             |unidade_medida|
+--------+---------+------------+--------------------------+------------------+--------------+
|6.0     |-73.5    |2025-12-01  |Poluição do ar - NO₂ (ppb)|7.681015886863419 |ppb           |
|6.0     |-72.75   |2025-12-01  |Poluição do ar - NO₂ (ppb)|6.558746342766711 |ppb           |
|6.0     |-72.0    |2025-12-01  |Poluição do ar - NO₂ (ppb)|1.9492423673422823|ppb           |
|6.0     |-71.25   |2025-12-01  |Poluição do ar - NO₂ (ppb)|0.5570433600500193|ppb           |
|6.0     |-70.5    |2025-12-01  |Poluição do ar - NO₂ (ppb)|0.2977518196830857|ppb

In [8]:
os.remove(r"C:\Marco Conti\Projetos\MAIS-v2\Poluicao\{file_name}".format(file_name = ret_download))